--- 
### 11.2 Inisialisasi SparkSession & Dataset Simulasi

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)


26/11/12 17:50:57 WARN Utils: Your hostname, raka-Aspire-A314-32 resolves to a loopback address: 127.0.1.1; using 10.90.70.61 instead (on interface wlp3s0)
26/11/12 17:50:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/12 17:50:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
np.random.seed(42)
n = 2000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "rating": np.random.randint(1, 6, size=n),
})
# Total pendapatan dipengaruhi unit & harga (dengan sedikit variasi acak, seperti data real world)
data["pendapatan"] = data["unit_terjual"] * data["harga_satuan"] + np.random.normal(0, 20000, size=n)

df = spark.createDataFrame(data)
df.show(5)
print("Total baris:", df.count())


+------------+------------+------------+----------+------+------------------+
|unit_terjual|harga_satuan|    kategori|      kota|rating|        pendapatan|
+------------+------------+------------+----------+------+------------------+
|           7|      150000|     Fashion|  Magelang|     4| 997913.0612957058|
|           4|      100000|Rumah Tangga|  Semarang|     4| 389490.8317158621|
|          13|      150000|  Elektronik|Yogyakarta|     4|1936047.6709878189|
|          11|      100000|Rumah Tangga|  Magelang|     5|1080918.9109179082|
|           8|      150000|     Makanan|  Semarang|     2|1191696.4943767258|
+------------+------------+------------+----------+------+------------------+
only showing top 5 rows



Total baris: 2000


--- 
### 11.3 Feature Engineering: StringIndexer & VectorAssembler

In [3]:
from pyspark.ml.feature import StringIndexer

indexer_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
indexer_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")

df_indexed = indexer_kategori.fit(df).transform(df)
df_indexed = indexer_kota.fit(df_indexed).transform(df_indexed)

df_indexed.select("kategori", "kategori_idx", "kota", "kota_idx").show(8)


+------------+------------+----------+--------+
|    kategori|kategori_idx|      kota|kota_idx|
+------------+------------+----------+--------+
|     Fashion|         4.0|  Magelang|     1.0|
|Rumah Tangga|         1.0|  Semarang|     2.0|
|  Elektronik|         3.0|Yogyakarta|     0.0|
|Rumah Tangga|         1.0|  Magelang|     1.0|
|     Makanan|         0.0|  Semarang|     2.0|
|   Kesehatan|         2.0|Yogyakarta|     0.0|
|Rumah Tangga|         1.0|      Solo|     3.0|
|   Kesehatan|         2.0|  Semarang|     2.0|
+------------+------------+----------+--------+
only showing top 8 rows



In [4]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_idx", "kota_idx", "rating"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "pendapatan").show(5, truncate=False)


+---------------------------+------------------+
|features                   |pendapatan        |
+---------------------------+------------------+
|[7.0,150000.0,4.0,1.0,4.0] |997913.0612957058 |
|[4.0,100000.0,1.0,2.0,4.0] |389490.8317158621 |
|[13.0,150000.0,3.0,0.0,4.0]|1936047.6709878189|
|[11.0,100000.0,1.0,1.0,5.0]|1080918.9109179082|
|[8.0,150000.0,0.0,2.0,2.0] |1191696.4943767258|
+---------------------------+------------------+
only showing top 5 rows



--- 
### 11.4 Melatih Model Regresi Linear

In [5]:
# randomSplit membagi data secara acak — 80% untuk latihan (training), 20% untuk pengujian (testing)
train, test = df_final.randomSplit([0.8, 0.2], seed=42)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")


Jumlah data training : 1642


Jumlah data testing   : 358


In [6]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="pendapatan")
model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print(f"Intercept: {model.intercept:,.2f}")


Model berhasil dilatih!
Koefisien tiap fitur: [80447.77618777308,7.613667911708261,2806.280730215923,5314.906749700611,-4814.096787233657]
Intercept: -607,672.18


In [7]:
predictions = model.transform(test)
predictions.select("unit_terjual", "harga_satuan", "kategori", "kota", "pendapatan", "prediction").show(10)


+------------+------------+------------+----------+------------------+-------------------+
|unit_terjual|harga_satuan|    kategori|      kota|        pendapatan|         prediction|
+------------+------------+------------+----------+------------------+-------------------+
|           1|       25000|  Elektronik|  Magelang|15220.154208430495| -342405.3489269009|
|           1|       25000|     Fashion|  Semarang|19301.753042450702|-329470.06465975073|
|           1|       25000|   Kesehatan|  Magelang| 6863.845479462816|-330769.33929541585|
|           1|       25000|Rumah Tangga|  Magelang|64514.717361452844| -338389.7168128654|
|           1|       50000|     Fashion|  Semarang| 91810.59803943225|-134314.27007981052|
|           1|       50000|   Kesehatan|  Semarang| 54842.66311348053|-139926.83154024236|
|           1|       50000|     Makanan|  Magelang| 45646.28680997743|-160482.49332484213|
|           1|       50000|Rumah Tangga|  Magelang| 37680.87612973162| -143233.9222329252|

--- 
### 11.5 Evaluasi Model Regresi

In [8]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")


RMSE : Rp 159,448
R²   : 0.8955  (89.5% variasi data berhasil dijelaskan model)


--- 
### Menutup SparkSession

In [9]:
spark.stop()
print("SparkSession ditutup secara bersih.")


SparkSession ditutup secara bersih.
